# MediConnect — AI-Powered Emergency Healthcare Platform

## Gemma-4 Good Hackathon Submission

### Problem Statement
In medical emergencies, people waste critical time searching for the right doctor. MediConnect solves this with AI-powered symptom analysis + real-time doctor availability.

### Key Features
1. **AI Symptom Analyzer** — urgency detection + specialist routing
2. **Emergency Mode** — nearest doctor in one tap
3. **Doctor Finder** — distance-sorted with live availability
4. **Trust System** — AI fake review detection + scoring
5. **Appointment Booking** — clinic, online, home visit

In [ ]:
# Install dependencies
!pip install requests -q

In [ ]:
import json
import requests

# Add your Groq API key here
GROQ_API_KEY = 'your_groq_key_here'

def analyze_symptoms(symptoms, age, gender):
    prompt = f"""You are an expert medical triage assistant.
Patient: Age {age}, Gender {gender}
Symptoms: {symptoms}

Respond ONLY with valid JSON:
{{
  \"possible_conditions\": [\"condition1\", \"condition2\"],
  \"urgency_level\": \"low\",
  \"recommended_doctor_type\": \"specialty\",
  \"advice\": \"actionable advice\",
  \"go_to_emergency\": false
}}"""

    response = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {GROQ_API_KEY}',
            'Content-Type': 'application/json'
        },
        json={
            'model': 'llama-3.1-8b-instant',
            'messages': [
                {'role': 'system', 'content': 'You are a medical triage assistant. Always respond with valid JSON only. No markdown.'},
                {'role': 'user', 'content': prompt}
            ],
            'temperature': 0.2,
            'max_tokens': 400
        }
    )
    data = response.json()
    raw = data['choices'][0]['message']['content']
    return json.loads(raw[raw.find('{'):raw.rfind('}')+1])

print('AI Service initialized successfully!')

In [ ]:
# Test Case 1 — Critical Emergency
print('TEST 1: Critical Emergency')
print('=' * 40)
result = analyze_symptoms('severe chest pain and difficulty breathing', 45, 'male')
print(json.dumps(result, indent=2))

In [ ]:
# Test Case 2 — Medium Urgency
print('TEST 2: Medium Urgency')
print('=' * 40)
result = analyze_symptoms('fever 38.5C, headache and body ache for 2 days', 28, 'female')
print(json.dumps(result, indent=2))

In [ ]:
# Test Case 3 — Low Urgency
print('TEST 3: Low Urgency')
print('=' * 40)
result = analyze_symptoms('mild cold, runny nose and slight sore throat', 22, 'male')
print(json.dumps(result, indent=2))

In [ ]:
# Fake Review Detection Demo
import math

def detect_fake_review(review_text, rating):
    words = review_text.strip().split()
    if len(words) < 12:
        return True, 'Too short'
    if review_text.count('!') >= 2:
        return True, 'Excessive punctuation'
    spam = ['best doctor ever', 'highly recommend', '100% recommend', 'amazing doctor']
    if any(p in review_text.lower() for p in spam):
        return True, 'Generic spam phrase detected'
    medical_words = ['treatment', 'diagnosis', 'consultation', 'symptoms', 'explained', 'medicine']
    if not any(w in review_text.lower() for w in medical_words):
        return True, 'No medical context'
    return False, 'Looks genuine'

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return round(R * 2 * math.asin(math.sqrt(a)), 1)

def calculate_trust_score(rating, experience_years, review_count):
    rating_score     = ((rating - 1) / 4) * 100
    experience_score = min(experience_years / 20, 1.0) * 100
    review_score     = min(review_count / 10, 1.0) * 100
    return round(rating_score * 0.5 + experience_score * 0.3 + review_score * 0.2)

print('FAKE REVIEW DETECTION DEMO')
print('=' * 40)
tests = [
    ('Best doctor ever!! Amazing!! 100% recommend!!!', 5),
    ('Dr. Roy was very thorough during my consultation. She explained the diagnosis clearly and the treatment worked well after 3 days.', 5),
    ('ok good', 4),
]
for review, rating in tests:
    is_fake, reason = detect_fake_review(review, rating)
    status = 'FLAGGED' if is_fake else 'APPROVED'
    print(f'[{status}] {reason}')
    print(f'Review: "{review[:60]}..."')
    print()

In [ ]:
# Doctor Ranking Demo
print('DOCTOR TRUST SCORE DEMO')
print('=' * 40)

doctors = [
    {'name': 'Dr. Arjun Sharma',    'rating': 4.8, 'experience': 12, 'reviews': 8},
    {'name': 'Dr. Priya Banerjee',  'rating': 4.7, 'experience': 15, 'reviews': 5},
    {'name': 'Dr. Meena Ghosh',     'rating': 4.9, 'experience': 20, 'reviews': 12},
    {'name': 'Dr. Fatima Khatun',   'rating': 4.7, 'experience': 18, 'reviews': 3},
]

for doc in doctors:
    score = calculate_trust_score(doc['rating'], doc['experience'], doc['reviews'])
    print(f"{doc['name']}: Trust Score = {score}/100")

print()
print('DISTANCE CALCULATION DEMO (from Salt Lake, Kolkata)')
print('=' * 40)
user_lat, user_lon = 22.5726, 88.3639
locations = [
    ('Dr. Priya Banerjee - Park Street', 22.5513, 88.3523),
    ('Dr. Meena Ghosh - Dum Dum',        22.6488, 88.4272),
    ('Dr. Sanjay Mukherjee - Ballygunge', 22.5264, 88.3631),
]
for name, lat, lon in locations:
    dist = haversine(user_lat, user_lon, lat, lon)
    print(f'{name}: {dist} km away')

## Architecture

```
User (React Frontend)
        ↓
FastAPI Backend (Python)
        ↓
AI Layer (Llama 3.1 via Groq API)
        ↓
SQLite Database
```

## Impact
- Reduces emergency response time significantly
- Makes healthcare accessible and transparent
- Builds trust through AI-verified reviews
- 100% free to use — no paid APIs required

## Tech Stack
- **Frontend:** React + Vite + Tailwind
- **Backend:** Python + FastAPI
- **Database:** SQLite
- **AI:** Llama 3.1 8B via Groq (free tier)
- **Maps:** Google Maps Directions API